---

# NLP Modelling Pipeline Template

*A dataset-agnostic text classification and regression pipeline: an EDA-informed TF-IDF baseline followed by selectable transformer encoders (IndoBERT, RoBERTa, XLM-R, ModernBERT, DeBERTa-v3, and more), ensembling, honest holdout evaluation, interpretation, and export to `pipeline-output/baseline/` and `pipeline-output/final/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Toolkit**](#3)
4. [**EDA-Informed Decisions**](#4)
5. [**Data Preparation**](#5)
6. [**Baseline: TF-IDF + Linear Model**](#6)
7. [**Transformer Setup**](#7)
8. [**Transformer Fine-Tuning**](#8)
9. [**Ensemble and Model Selection**](#9)
10. [**Final Evaluation on Holdout**](#10)
11. [**Model Interpretation**](#11)
12. [**Error Analysis**](#12)
13. [**Predictions and Export**](#13)


---

# Introduction <a name="1"></a>

---

## Overview

*Fine-tuned transformer encoders are the state of the art for text classification, but they are expensive and easy to misconfigure. This notebook makes every expensive choice (encoder, sequence length, class weighting, duplicate handling) traceable to the EDA, measures every transformer against a strong TF-IDF baseline, and reports an unbiased holdout score.*

## Aim

*The notebook trains a TF-IDF (word and character n-grams) linear baseline that always runs, fine-tunes any list of transformer encoders on shared cross-validation folds, blends them, selects the best candidate by cross-validated score, and evaluates it once on a held-out set. It supports binary classification, multiclass classification, and regression on text, optionally with a second text column for sentence-pair tasks.*

## How to Configure

*Everything is set in the `Settings` class (section 2). Settings left on `'auto'` are filled from `eda-output/eda_decisions.json`, written by the NLP EDA notebook, or computed on the fly when that file does not exist.*

| Setting | Purpose | Example |
|---|---|---|
| `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH`, `TEST_*_PATH` | Training file and optional unlabeled test file | `'/kaggle/input/comp/train.csv'` |
| `TEXT_COL`, `TEXT_PAIR_COL`, `LABEL_COL` | Text, optional second text, target | `'review'`, `None`, `'sentiment'` |
| `EDA_DIR` | Folder with `eda_decisions.json` | `'eda-output'` |
| `TASK`, `ORDINAL_ORDER`, `MULTILABEL_SEP` | `'auto'`, `'classification'`, `'regression'`, `'ordinal'` (QWK), or `'multilabel'`; class order for ordinal strings; separator when one column holds several labels (or pass a list of 0/1 columns as `LABEL_COL`) | `'ordinal'`, `['low', 'mid', 'high']`, `'|'` |
| `TUNE_DECISION` | Tune the label rule on out-of-fold predictions: binary threshold, multiclass class scales, ordinal cut-points, one threshold per label for multi-label | `True` |
| `ID_COL`, `SUBMISSION_FORMAT` | Id column kept in `submission.csv`; `'label'`, `'proba'`, or `'auto'` (probabilities for ROC AUC and log loss) | `'review_id'`, `'auto'` |
| `MODELS` | List of keys from `MODEL_REGISTRY`, or `['auto']` | `['indobert', 'xlm-roberta']` |
| `MAX_LENGTH`, `TRUNCATION` | Token budget (`'auto'` from EDA) and `'head'` or `'head_tail'` | `256`, `'head_tail'` |
| `EPOCHS`, `LR`, `BATCH_SIZE`, `TRAIN_FOLDS` | Training budget; `TRAIN_FOLDS` limits which CV folds are fine-tuned | `3`, `2e-5`, `16`, `[0]` |

*Model keys available in `MODEL_REGISTRY`:*

| Key | Checkpoint | Best for |
|---|---|---|
| `indobert` | `indobenchmark/indobert-base-p1` | Indonesian (formal and informal) |
| `indobert-large` | `indobenchmark/indobert-large-p1` | Indonesian, more capacity |
| `indolem` | `indolem/indobert-base-uncased` | Indonesian, uncased |
| `indoroberta` | `flax-community/indonesian-roberta-base` | Indonesian RoBERTa |
| `roberta` | `FacebookAI/roberta-base` | English |
| `twitter-roberta` | `cardiffnlp/twitter-roberta-base-sentiment-latest` | English social media and sentiment |
| `deberta-v3` | `microsoft/deberta-v3-base` | English, strongest base encoder |
| `modernbert` | `answerdotai/ModernBERT-base` | English, long context (up to 8k tokens), fast |
| `xlm-roberta` | `FacebookAI/xlm-roberta-base` | Multilingual or code-mixed |
| `mdeberta` | `microsoft/mdeberta-v3-base` | Multilingual, DeBERTa quality |
| `multilingual-e5` | `intfloat/multilingual-e5-base` | Multilingual, strong sentence semantics |
| `distilbert` | `distilbert/distilbert-base-uncased` | Fast English experiments |
| `bert-tiny` | `google/bert_uncased_L-2_H-128_A-2` | Smoke test only |

*Any other Hugging Face checkpoint can be used by adding one line to `MODEL_REGISTRY`.*

## Metric

***Macro F1** is the default classification metric because it weights every class equally, which matters for the imbalanced labels typical of reviews and comments. **RMSE** is the default for regression. Accuracy, weighted F1, ROC AUC, log loss, MAE, and R-squared are reported alongside.*

$$\text{Macro F1} = \frac{1}{K}\sum_{k=1}^{K} \frac{2 \cdot P_k \cdot R_k}{P_k + R_k} \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

## Dataset

*The demo uses the lyrics of the DAC PRS ITS 2025 dataset with `track_genre` as the label, restricted to the most frequent genres through `TOP_K_CLASSES`. Replace the paths and columns with the competition data on the day.*

```
referensi/
└── data.csv
     ├── lyrics        (text)
     └── track_genre   (label)

eda-output/
└── eda_decisions.json   (written by eda_nlp.ipynb)
```

## Approach: EDA-Informed Baseline-to-Transformer Pipeline

*A holdout set is separated first and touched only once at the end. All models share the same cross-validation folds, grouped so that identical texts never straddle a fold boundary. The transformer head mean-pools the last hidden layer, which is robust across encoder families.*

```
eda_decisions.json --> resolved settings (model family, max_length, cleaning, weights)
Raw text -> clean -> dedup -> holdout split -> grouped CV folds
    |
    +-- Baseline     : TF-IDF word 1-2 + char 2-5 -> Logistic / Ridge   -> pipeline-output/baseline/
    +-- Transformers : tokenizer (head or head+tail truncation)
    |                   -> encoder -> mean pooling -> dropout -> linear head
    |                   AdamW + cosine warmup, mixed precision, early stopping
    +-- Ensemble     : non-negative weight blend on out-of-fold predictions
         |
    Best by CV -> holdout evaluation -> interpretation -> error analysis
         -> pipeline-output/final/ (submission, predictions, report)
```

---

# Initialization <a name="2"></a>

---

## Environment Setup

Transformer fine-tuning needs a GPU. On Kaggle, enable a T4 x2 or P100 accelerator. A base model on 10,000 texts of 256 tokens takes roughly 3 to 5 minutes per epoch on a T4 with mixed precision. Without a GPU, set `MODELS = []` to run the baseline only.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn pyarrow openpyxl torch transformers sentencepiece protobuf

## Import Libraries

PyTorch and Transformers are imported defensively. If either is missing, the notebook still runs the TF-IDF baseline end to end.

In [ ]:
import gc
import os
import re
import html
import inspect
import json
import math
import time
import random
import hashlib
import warnings
import unicodedata
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import sparse
from scipy.optimize import minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, f1_score, log_loss,
                             mean_absolute_error, mean_squared_error, precision_recall_fscore_support,
                             r2_score, roc_auc_score)
from sklearn.model_selection import (GroupShuffleSplit, KFold, StratifiedGroupKFold, GroupKFold,
                                     StratifiedKFold, train_test_split)
from sklearn.preprocessing import LabelEncoder
from IPython.display import HTML, Markdown, display

AVAILABLE = {}
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    import transformers
    from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_cosine_schedule_with_warmup
    transformers.logging.set_verbosity_error()
    transformers.utils.logging.disable_progress_bar()
    AVAILABLE['torch'] = torch.__version__
    AVAILABLE['transformers'] = transformers.__version__
except Exception as e:
    AVAILABLE['torch'] = AVAILABLE['transformers'] = False
    print(f'[warn] torch / transformers unavailable: {str(e).splitlines()[0][:120]}')

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)
DEVICE = 'cpu'
if AVAILABLE['torch']:
    DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    if DEVICE == 'cuda':
        print(f'GPU: {torch.cuda.get_device_name(0)} x {torch.cuda.device_count()}')
print(f'Device: {DEVICE} | libraries: {AVAILABLE}')

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    if AVAILABLE['torch']:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

Paths, columns, EDA link, validation, baseline, and transformer hyperparameters are centralised here. `'auto'` values are resolved in section 4 from the EDA decisions file. `MODEL_OVERRIDES` sets per-model hyperparameters, for example a smaller batch for large models.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}
    EDA_DIR          = 'eda-output'

    # 2) Columns and task
    TEXT_COL      = 'auto'
    TEXT_PAIR_COL = None
    LABEL_COL     = 'auto'
    ID_COL        = 'track_id'
    SUBMISSION_FORMAT = 'auto'
    TASK          = 'auto'
    ORDINAL_ORDER = None
    MULTILABEL_SEP = None
    TOP_K_CLASSES = 6
    DEBUG_SAMPLE  = None

    # 3) Cleaning and validation
    REMOVE_PATTERNS = 'auto'
    DEDUP           = 'auto'
    GROUP_IDENTICAL = 'auto'
    CV_GROUP        = None
    N_FOLDS         = 5
    HOLDOUT_SIZE    = 0.15
    METRIC          = 'auto'
    CLASS_WEIGHTS   = 'auto'
    TUNE_DECISION   = True

    # 4) TF-IDF baseline
    TFIDF_MAX_FEATURES = 'auto'
    TFIDF_CHAR         = True
    BASELINE_C         = 4.0
    BASELINE_ALPHA     = 1.0

    # 5) Transformers
    MODELS          = ['auto']
    MODEL_REGISTRY  = {
        'indobert':        'indobenchmark/indobert-base-p1',
        'indobert-large':  'indobenchmark/indobert-large-p1',
        'indolem':         'indolem/indobert-base-uncased',
        'indoroberta':     'flax-community/indonesian-roberta-base',
        'roberta':         'FacebookAI/roberta-base',
        'twitter-roberta': 'cardiffnlp/twitter-roberta-base-sentiment-latest',
        'deberta-v3':      'microsoft/deberta-v3-base',
        'modernbert':      'answerdotai/ModernBERT-base',
        'xlm-roberta':     'FacebookAI/xlm-roberta-base',
        'mdeberta':        'microsoft/mdeberta-v3-base',
        'multilingual-e5': 'intfloat/multilingual-e5-base',
        'distilbert':      'distilbert/distilbert-base-uncased',
        'bert-tiny':       'google/bert_uncased_L-2_H-128_A-2',
    }
    MODEL_OVERRIDES = {'indobert-large': {'batch_size': 8, 'lr': 1e-5}}
    MAX_LENGTH      = 'auto'
    TRUNCATION      = 'head_tail'
    HEAD_RATIO      = 0.5
    POOLING         = 'mean'
    EPOCHS          = 3
    BATCH_SIZE      = 16
    GRAD_ACCUM      = 1
    LR              = 2e-5
    HEAD_LR_MULT    = 5.0
    WEIGHT_DECAY    = 0.01
    WARMUP_RATIO    = 0.1
    LABEL_SMOOTHING = 0.0
    DROPOUT         = 0.1
    EVALS_PER_EPOCH = 2
    PATIENCE        = 3
    MIXED_PRECISION = 'auto'
    GRAD_CHECKPOINT = False
    TRAIN_FOLDS     = [0]
    SAVE_WEIGHTS    = False
    ENSEMBLE        = True

    # 6) Outputs
    DATA_PATH    = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH    = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    OUTPUT_DIR   = Path('/kaggle/working/pipeline-output') if _ON_KAGGLE else Path('pipeline-output')
    BASELINE_DIR = OUTPUT_DIR / 'baseline'
    FINAL_DIR    = OUTPUT_DIR / 'final'

CFG = Settings()
for d in (CFG.BASELINE_DIR / 'figures', CFG.FINAL_DIR / 'figures', CFG.FINAL_DIR / 'oof', CFG.FINAL_DIR / 'models'):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Load Dataset

The path is resolved with the same Kaggle fallback as the other templates, the reader is chosen from the extension, and the optional test file is loaded.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.jsonl', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.jsonl' in suffixes:
        return pd.read_json(path, lines=True, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = [f for f in r.rglob('*') if f.is_file() and f.suffix.lower() in TABULAR_EXT
                     and not SKIP_DIRS & set(f.parts)]
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, CFG.TEST_PATH)
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found and no train file was discovered. Edit the path lines in Settings.'
if TEST_PATH is None:
    print('[warn] No test file configured or found in the dataset folder: submission.csv will not be written.')
raw = load_table(DATA_PATH, **CFG.READ_KWARGS)
raw_test = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
SAMPLE = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
print(f'Train: {raw.shape[0]:,} rows x {raw.shape[1]} cols' + (f' | Test: {raw_test.shape[0]:,} rows' if raw_test is not None else ''))

---

# Toolkit <a name="3"></a>

---

The plotting identity matches the other templates, and `save_fig` writes figures into the baseline or final output folder with a running number. The metric functions are shared by every model: regression predictions are values, classification predictions are probability matrices with one column per class.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = {'baseline': 0, 'final': 0}

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, stage='final'):
    FIG_COUNTER[stage] += 1
    root = CFG.BASELINE_DIR if stage == 'baseline' else CFG.FINAL_DIR
    fig.savefig(root / 'figures' / f'{FIG_COUNTER[stage]:02d}_{slug(name)}.png', facecolor='white')
    plt.show()
    plt.close(fig)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def bar_labels(ax, fmt='{:,.0f}', size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues], padding=3, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

def to_json(obj, path):
    path.write_text(json.dumps(obj, indent=2, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)), encoding='utf-8')

In [ ]:
GREATER_IS_BETTER = {'rmse': False, 'mae': False, 'r2': True, 'f1_macro': True, 'f1_weighted': True,
                     'accuracy': True, 'balanced_accuracy': True, 'roc_auc': True, 'logloss': False, 'f1': True,
                     'qwk': True, 'mae_class': False, 'f1_micro': True, 'f1_samples': True}
THRESHOLD, CLASS_SCALE, CUTS, ML_THRESH = [0.5], [None], [None], [None]
IS_ORD = IS_ML = False
ML_SEP = None

def to_label(proba):
    """Labels from probabilities: per-label thresholds (multi-label), threshold (binary), or scaled argmax (multiclass)."""
    if IS_ML:
        return (proba >= (ML_THRESH[0] if ML_THRESH[0] is not None else 0.5)).astype(int)
    if proba.shape[1] == 2:
        return (proba[:, 1] >= THRESHOLD[0]).astype(int)
    if CLASS_SCALE[0] is not None:
        return (proba * CLASS_SCALE[0]).argmax(axis=1)
    return proba.argmax(axis=1)

def to_ordinal(pred):
    cuts = CUTS[0] if CUTS[0] is not None else np.arange(N_CLASSES - 1) + 0.5
    return np.digitize(pred, cuts)

def ml_names(row, sep=None):
    return (sep or ML_SEP or '|').join(CLASS_NAMES[j] for j in np.where(np.asarray(row) > 0)[0])

def pred_labels(pred):
    """Final label strings for any task type."""
    if IS_ML:
        return np.array([ml_names(r) for r in to_label(pred)], dtype=object)
    if IS_ORD:
        return np.array(CLASS_NAMES, dtype=object)[to_ordinal(pred)]
    return LE.inverse_transform(to_label(pred))

def score_all(y_true, pred):
    if IS_ORD:
        yt, lab = np.asarray(y_true).astype(int), to_ordinal(pred)
        return {'qwk': float(cohen_kappa_score(yt, lab, weights='quadratic')), 'accuracy': float(accuracy_score(yt, lab)),
                'f1_macro': float(f1_score(yt, lab, average='macro')), 'mae_class': float(np.mean(np.abs(yt - lab))),
                'rmse': float(np.sqrt(mean_squared_error(yt, pred)))}
    if IS_REG:
        return {'rmse': float(np.sqrt(mean_squared_error(y_true, pred))), 'mae': float(mean_absolute_error(y_true, pred)),
                'r2': float(r2_score(y_true, pred))}
    if IS_ML:
        lab = to_label(pred)
        q = np.clip(pred, 1e-7, 1 - 1e-7)
        out = {'f1_micro': f1_score(y_true, lab, average='micro', zero_division=0),
               'f1_macro': f1_score(y_true, lab, average='macro', zero_division=0),
               'f1_samples': f1_score(y_true, lab, average='samples', zero_division=1),
               'logloss': float(-np.mean(y_true * np.log(q) + (1 - y_true) * np.log(1 - q)))}
        try:
            out['roc_auc'] = roc_auc_score(y_true, pred, average='macro')
        except ValueError:
            out['roc_auc'] = np.nan
        return {k: float(v) for k, v in out.items()}
    lab = to_label(pred)
    p = np.clip(pred, 1e-7, 1)
    p = p / p.sum(axis=1, keepdims=True)
    out = {'f1_macro': f1_score(y_true, lab, average='macro'), 'f1_weighted': f1_score(y_true, lab, average='weighted'),
           'accuracy': accuracy_score(y_true, lab), 'balanced_accuracy': balanced_accuracy_score(y_true, lab),
           'logloss': log_loss(y_true, p, labels=list(range(N_CLASSES)))}
    if N_CLASSES == 2:
        out['f1'] = f1_score(y_true, lab)
    try:
        out['roc_auc'] = roc_auc_score(y_true, p[:, 1]) if N_CLASSES == 2 else roc_auc_score(y_true, p, multi_class='ovr', labels=list(range(N_CLASSES)))
    except ValueError:
        out['roc_auc'] = np.nan
    return {k: float(v) for k, v in out.items()}

def primary(y_true, pred):
    return score_all(y_true, pred)[METRIC]

---

# EDA-Informed Decisions <a name="4"></a>

---

This section connects the EDA notebook to the pipeline. `eda_decisions.json` is loaded when it exists, and every setting on `'auto'` takes its value from it. When the file is missing (or describes a different dataset), the same quantities are computed directly from the data. The resulting table records each decision and its source (`user`, `eda`, or `computed`), ready to be quoted in the report's methodology chapter.

In [ ]:
eda_path = Path(CFG.EDA_DIR) / 'eda_decisions.json'
EDA = json.loads(eda_path.read_text()) if eda_path.exists() else {}
if EDA and Path(EDA.get('source', '')).name != DATA_PATH.name:
    print(f'[warn] EDA decisions were made on {EDA.get("source")}, not {DATA_PATH.name}; using them anyway')
print(f'EDA decisions file: {eda_path.resolve()} ({"found" if EDA else "not found, computing from data"})')
DECISIONS = []

def decide(name, user_value, eda_key, compute):
    if user_value != 'auto' and user_value != ['auto']:
        value, source = user_value, 'user'
    elif eda_key and EDA.get(eda_key) is not None:
        value, source = EDA[eda_key], 'eda'
    else:
        value, source = compute(), 'computed'
    DECISIONS.append({'setting': name, 'value': str(value)[:90], 'source': source})
    return value

def longest_text_col():
    obj = [c for c in raw.columns if not pd.api.types.is_numeric_dtype(raw[c])]
    return max(obj, key=lambda c: raw[c].dropna().astype(str).str.len().mean())

def guess_target(train, test):
    """Infer the target: the sample-submission target column, else the one train column missing from the test file."""
    if SAMPLE is not None and SAMPLE.shape[1] == 2 and SAMPLE.columns[1] in train.columns:
        return SAMPLE.columns[1]
    if test is not None:
        only_train = [c for c in train.columns if c not in test.columns]
        if len(only_train) == 1:
            return only_train[0]
    return None

TEXT_COL = decide('TEXT_COL', CFG.TEXT_COL, 'text_col', longest_text_col)
LABEL_COL = decide('LABEL_COL', CFG.LABEL_COL, 'label_col', lambda: guess_target(raw, raw_test))
LABEL_COLS = list(LABEL_COL) if isinstance(LABEL_COL, (list, tuple)) else [LABEL_COL]
if LABEL_COLS == [None] and SAMPLE is not None and SAMPLE.shape[1] > 2 and all(c in raw.columns for c in SAMPLE.columns[1:]):
    LABEL_COLS = list(SAMPLE.columns[1:])
assert all(c in raw.columns for c in LABEL_COLS), f'Set LABEL_COL; available columns: {list(raw.columns)}'
ML_COLUMNS = None
if len(LABEL_COLS) > 1:
    # several 0/1 columns are one multi-label target
    ML_COLUMNS = LABEL_COLS
    on = raw[LABEL_COLS].astype(str).isin(['1', '1.0', 'True', 'true']).values
    raw['__labels__'] = ['|'.join(c for c, hit in zip(LABEL_COLS, row) if hit) for row in on]
    LABEL_COL, ML_SEP = '__labels__', '|'
    print(f'Multi-label target built from {len(LABEL_COLS)} binary columns')
else:
    LABEL_COL = LABEL_COLS[0]
    ML_SEP = CFG.MULTILABEL_SEP
REMOVE_PATTERNS = decide('REMOVE_PATTERNS', CFG.REMOVE_PATTERNS, 'remove_patterns', lambda: [])
REMOVE_RES = [re.compile(p) for p in REMOVE_PATTERNS]

## Language, Sequence Length, and Vocabulary

When the EDA file is absent, language is estimated with the same stopword heuristic as the EDA, and the word-length percentile and vocabulary coverage are computed on a sample. The language picks the default encoder family: IndoBERT for Indonesian, DeBERTa-v3 for English, and XLM-RoBERTa for mixed or other languages.

In [ ]:
EN_SW = set('the and to of a in is it you that for on with this was are be have not but my they'.split())
ID_SW = set('yang dan di ke dari ini itu dengan untuk tidak ada saya aku kamu nya gak ga aja juga sudah bisa'.split())
sample_texts = raw[TEXT_COL].dropna().astype(str).sample(min(3000, raw[TEXT_COL].notna().sum()), random_state=CFG.SEED)
sample_words = [re.findall(r'[^\W\d_]+', t.lower()) for t in sample_texts]

def compute_language():
    en = np.mean([sum(w in EN_SW for w in t) / max(len(t), 1) for t in sample_words])
    idn = np.mean([sum(w in ID_SW for w in t) / max(len(t), 1) for t in sample_words])
    return 'id' if idn > 1.5 * en else 'en' if en > 1.5 * idn else 'multi'

def compute_max_length():
    p95 = np.percentile([len(t) for t in sample_words], 95)
    return int(min(512, 2 ** np.ceil(np.log2(max(p95 * 1.3, 16)))))

def compute_tfidf_size():
    vocab = pd.Series([w for t in sample_words for w in t]).value_counts()
    cov = np.cumsum(vocab.values) / vocab.values.sum()
    return int(min(200_000, max(20_000, np.searchsorted(cov, 0.95) * 3)))

LANGUAGE = decide('LANGUAGE', 'auto', 'language', compute_language)
MAX_LENGTH = int(decide('MAX_LENGTH', CFG.MAX_LENGTH, 'max_length', compute_max_length))
TFIDF_MAX_FEATURES = int(decide('TFIDF_MAX_FEATURES', CFG.TFIDF_MAX_FEATURES, 'tfidf_max_features', compute_tfidf_size))
FAMILY = {'id': 'indobert', 'en': 'deberta-v3', 'multi': 'xlm-roberta'}
MODELS = decide('MODELS', CFG.MODELS, 'recommended_model', lambda: FAMILY[LANGUAGE])
MODELS = [MODELS] if isinstance(MODELS, str) else list(MODELS)
if EDA.get('noise_patterns'):
    print('EDA noise patterns above 1% of documents:', EDA['noise_patterns'])

---

# Data Preparation <a name="5"></a>

---

Text cleaning for transformers is deliberately light, because pretrained tokenizers already handle casing, punctuation, and rare words, and aggressive cleaning destroys signal such as emphasis or emoji. Only the configured `REMOVE_PATTERNS`, URLs, HTML entities, and redundant whitespace are removed. The TF-IDF baseline lowercases internally.

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')

def clean_text(t):
    t = html.unescape(str(t))
    for rx in REMOVE_RES:
        t = rx.sub(' ', t)
    t = URL_RE.sub(' URL ', t)
    t = unicodedata.normalize('NFKC', t)
    return re.sub(r'\s+', ' ', t).strip()

def build_frame(frame, has_label=True):
    out = pd.DataFrame({'text': frame[TEXT_COL].fillna('').map(clean_text)})
    if CFG.TEXT_PAIR_COL:
        out['text_pair'] = frame[CFG.TEXT_PAIR_COL].fillna('').map(clean_text)
    if CFG.ID_COL and CFG.ID_COL in frame.columns:
        out['id'] = frame[CFG.ID_COL].values
    if CFG.CV_GROUP and CFG.CV_GROUP in frame.columns:
        out['group'] = frame[CFG.CV_GROUP].astype(str).values
    if has_label:
        out['y_raw'] = frame[LABEL_COL].values
    return out

data = build_frame(raw.dropna(subset=[LABEL_COL]))
test = build_frame(raw_test, has_label=False) if raw_test is not None else None
data = data[data['text'].str.len() > 0].reset_index(drop=True)
print(f'Rows with text and label: {len(data):,}')
display(data.head(3))

## Target Preparation

The task is inferred from the label: strings or at most 20 distinct numeric values mean classification, otherwise regression. For long-tailed label sets, `TOP_K_CLASSES` keeps only the most frequent classes, and classes too small to appear in every fold are dropped. `DEBUG_SAMPLE` subsamples rows for a fast dry run.

In [ ]:
if CFG.TASK == 'auto':
    is_num = pd.api.types.is_numeric_dtype(data['y_raw'])
    TASK = ('multilabel' if ML_SEP else 'ordinal' if CFG.METRIC == 'qwk'
            else 'classification' if (not is_num or data['y_raw'].nunique() <= 20) else 'regression')
else:
    TASK = CFG.TASK
if TASK == 'multilabel' and not ML_SEP:
    raise ValueError('TASK = multilabel needs MULTILABEL_SEP or a list of 0/1 columns as LABEL_COL')
IS_ML = TASK == 'multilabel'
IS_ORD = TASK == 'ordinal'
IS_REG = TASK in ('regression', 'ordinal')

def debug_subsample(frame, strat=None):
    if CFG.DEBUG_SAMPLE and len(frame) > CFG.DEBUG_SAMPLE:
        frame = frame.sample(CFG.DEBUG_SAMPLE, random_state=CFG.SEED) if strat is None else \
            train_test_split(frame, train_size=CFG.DEBUG_SAMPLE, stratify=strat, random_state=CFG.SEED)[0]
        print(f'[debug] subsampled to {len(frame):,} rows')
    return frame.reset_index(drop=True)

def ordinal_order(values):
    if CFG.ORDINAL_ORDER:
        return [str(v) for v in CFG.ORDINAL_ORDER]
    uniq = pd.Series(values.dropna().astype(str).unique())
    as_num = pd.to_numeric(uniq, errors='coerce')
    return list(uniq.iloc[np.argsort(as_num.values)]) if as_num.notna().all() else sorted(uniq)

Y_COLS = None
if IS_ML:
    tags = data['y_raw'].fillna('').astype(str).map(lambda v: {x.strip() for x in v.split(ML_SEP) if x.strip()})
    counts = pd.Series([x for ts in tags for x in ts]).value_counts()
    ML_LABELS = sorted(counts[counts >= 2 * CFG.N_FOLDS].index)
    rare = sorted(set(counts.index) - set(ML_LABELS))
    if rare:
        print(f'[warn] labels with fewer than {2 * CFG.N_FOLDS} positives are not modelled: {rare[:20]}')
    data['y_raw'] = tags.map(lambda ts: ML_SEP.join(sorted(x for x in ts if x in ML_LABELS)))
    data = debug_subsample(data)
    CLASS_NAMES, N_CLASSES = ML_LABELS, len(ML_LABELS)
    Y_COLS = [f'__y{j}' for j in range(N_CLASSES)]
    for j, lab in enumerate(ML_LABELS):
        data[Y_COLS[j]] = data['y_raw'].map(lambda v, lab=lab: int(lab in v.split(ML_SEP)))
    data['y'] = LabelEncoder().fit_transform(data['y_raw'])
    share = data[Y_COLS].mean()
    IMBALANCE = float(share.max() / max(share.min(), 1e-9))
    print(f'Multi-label: {N_CLASSES} labels, {data[Y_COLS].sum(axis=1).mean():.2f} labels per text, '
          f'{(data[Y_COLS].sum(axis=1) == 0).mean():.1%} texts without any label')
elif IS_ORD:
    order = ordinal_order(data['y_raw'])
    rank = {v: i for i, v in enumerate(order)}
    known = data['y_raw'].astype(str).isin(rank)
    if (~known).any():
        print(f'[warn] dropping {int((~known).sum()):,} rows whose label is not in ORDINAL_ORDER')
        data = data[known].reset_index(drop=True)
    data = debug_subsample(data, data['y_raw'].astype(str))
    data['y'] = data['y_raw'].astype(str).map(rank).astype(float)
    CLASS_NAMES, N_CLASSES, IMBALANCE = order, len(order), None
    CUTS[0] = np.arange(N_CLASSES - 1) + 0.5
    print(f'Ordinal target with classes {CLASS_NAMES}: trained as regression on the rank, labels from tuned cut-points')
elif not IS_REG:
    data['y_raw'] = data['y_raw'].astype(str)
    if CFG.TOP_K_CLASSES:
        keep = data['y_raw'].value_counts().index[:CFG.TOP_K_CLASSES]
        data = data[data['y_raw'].isin(keep)].reset_index(drop=True)
    data = debug_subsample(data, data['y_raw'])
    vc = data['y_raw'].value_counts()
    rare = vc[vc < 2 * CFG.N_FOLDS].index.tolist()
    if rare:
        print(f'[warn] dropping classes with fewer than {2 * CFG.N_FOLDS} rows: {rare}')
        data = data[~data['y_raw'].isin(rare)].reset_index(drop=True)
    LE = LabelEncoder().fit(data['y_raw'])
    data['y'] = LE.transform(data['y_raw'])
    CLASS_NAMES = list(LE.classes_)
    N_CLASSES = len(CLASS_NAMES)
    counts = np.bincount(data['y'])
    IMBALANCE = float(counts.max() / counts.min())
else:
    data = debug_subsample(data)
    data['y'] = data['y_raw'].astype(float)
    N_CLASSES, CLASS_NAMES, IMBALANCE = 1, None, None

METRIC = decide('METRIC', CFG.METRIC, None,
                lambda: 'f1_micro' if IS_ML else 'qwk' if IS_ORD else 'rmse' if IS_REG else 'f1_macro')
USE_CW = bool(decide('CLASS_WEIGHTS', CFG.CLASS_WEIGHTS, None, lambda: bool(not IS_REG and IMBALANCE > 3)))
print(f'Task: {TASK} | classes: {N_CLASSES} | imbalance: {IMBALANCE} | metric: {METRIC} | class weights: {USE_CW}')

## Duplicates and Grouping

Rows with identical cleaned text *and* identical label are dropped, because they only inflate validation scores. Identical texts with different labels are kept, since they are genuine label noise, but every copy of a text is assigned to the same group, so a text never appears in both a training and a validation fold. Both behaviours follow the EDA when set to `'auto'`.

In [ ]:
data['text_key'] = [hashlib.md5(re.sub(r'\W+', ' ', t.lower()).strip().encode()).hexdigest() for t in data['text']]
DEDUP = decide('DEDUP', CFG.DEDUP, None, lambda: 'exact' if (EDA.get('duplicate_share', 0) > 0.01 or data['text_key'].duplicated().mean() > 0.01) else 'none')
GROUP_IDENTICAL = bool(decide('GROUP_IDENTICAL', CFG.GROUP_IDENTICAL, 'group_identical_texts',
                              lambda: bool(data['text_key'].duplicated().mean() > 0.005)))
if DEDUP == 'exact':
    n0 = len(data)
    data = data.drop_duplicates(subset=['text_key', 'y']).reset_index(drop=True)
    print(f'Removed {n0 - len(data):,} rows duplicated in both text and label')
conflicts = data.groupby('text_key')['y'].nunique()
print(f'Texts appearing with more than one label (label noise): {(conflicts > 1).sum():,}')
if 'group' in data.columns:
    GROUPS = data['group'].values
elif GROUP_IDENTICAL:
    GROUPS = data['text_key'].values
else:
    GROUPS = None

decision_table = pd.DataFrame(DECISIONS).set_index('setting')
decision_table.to_csv(CFG.FINAL_DIR / 'decisions.csv')
display(decision_table)

## Holdout Split and Folds

The holdout is carved out first with the same grouping and stratification as the folds. The remaining development set is split into `N_FOLDS` folds that every model shares.

In [ ]:
idx = np.arange(len(data))
y_all = data['y'].values
strat = None if IS_ML else y_all.astype(int) if IS_ORD else (y_all if not IS_REG else None)
if CFG.HOLDOUT_SIZE:
    if GROUPS is not None:
        if strat is not None:
            sgk = StratifiedGroupKFold(n_splits=max(2, int(round(1 / CFG.HOLDOUT_SIZE))), shuffle=True, random_state=CFG.SEED)
            dev_idx, hold_idx = next(sgk.split(idx, strat, GROUPS))
        else:
            dev_idx, hold_idx = next(GroupShuffleSplit(1, test_size=CFG.HOLDOUT_SIZE, random_state=CFG.SEED).split(idx, y_all, GROUPS))
    else:
        dev_idx, hold_idx = train_test_split(idx, test_size=CFG.HOLDOUT_SIZE, stratify=strat, random_state=CFG.SEED)
else:
    dev_idx, hold_idx = idx, np.array([], dtype=int)
dev = data.iloc[dev_idx].reset_index(drop=True)
hold = data.iloc[hold_idx].reset_index(drop=True)
y_dev, y_hold = (dev[Y_COLS].values, hold[Y_COLS].values) if IS_ML else (dev['y'].values, hold['y'].values)
S_DEV = strat[dev_idx] if strat is not None else None
g_dev = GROUPS[dev_idx] if GROUPS is not None else None
HAS_HOLD = len(hold) > 0

if g_dev is not None:
    splitter = StratifiedGroupKFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED) if S_DEV is not None else GroupKFold(CFG.N_FOLDS)
    FOLDS = list(splitter.split(np.arange(len(dev)), S_DEV if S_DEV is not None else np.zeros(len(dev)), g_dev))
else:
    splitter = StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED) if S_DEV is not None else KFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    FOLDS = list(splitter.split(np.arange(len(dev)), S_DEV))
if GROUPS is not None and HAS_HOLD:
    print(f'Group overlap dev/holdout: {len(set(GROUPS[dev_idx]) & set(GROUPS[hold_idx]))}')
print(f'Dev {len(dev):,} | holdout {len(hold):,} | test {0 if test is None else len(test):,} | {CFG.N_FOLDS} folds of ~{len(FOLDS[0][1]):,} rows')

if not IS_REG and not IS_ML:
    share = pd.DataFrame({'dev': pd.Series(y_dev).value_counts(normalize=True), 'holdout': pd.Series(y_hold).value_counts(normalize=True)}).fillna(0)
    share.index = [CLASS_NAMES[i] for i in share.index]
    share = share.sort_values('dev')
    fig, ax = plt.subplots(figsize=(10, 0.35 * len(share) + 1.8))
    ax.barh(np.arange(len(share)) - 0.2, share['dev'], height=0.4, color=PRIMARY, label='dev')
    ax.barh(np.arange(len(share)) + 0.2, share['holdout'], height=0.4, color=ACCENT, label='holdout')
    ax.set_yticks(range(len(share)))
    ax.set_yticklabels([short(i, 24) for i in share.index])
    ax.legend()
    suptitle(fig, 'Class share in dev and holdout', f'imbalance {IMBALANCE:.1f}x; grouped by identical text: {GROUP_IDENTICAL}')
    save_fig(fig, 'class_share', 'final')

#### Insights

> *Fill in after running.* List the resolved decisions and their sources. State how many duplicates were removed and how much label noise remains, because the latter bounds the best achievable score.

---

# Baseline: TF-IDF + Linear Model <a name="6"></a>

---

The baseline is a TF-IDF representation of word unigrams and bigrams plus character 2-5 grams (which are robust to typos, slang, and morphology, important for informal Indonesian), fed to Logistic Regression for classification or Ridge regression for numeric targets. It trains in seconds on CPU, is fully interpretable, and is often within a few points of a transformer, so it is the bar every transformer must clear. The vectorisers are unsupervised, so they are fitted once on the development texts (no labels are involved) and only the linear model is refitted inside each fold, which makes the baseline fast even on long documents.

In [ ]:
def combine(frame):
    if CFG.TEXT_PAIR_COL:
        return (frame['text'] + ' || ' + frame['text_pair']).tolist()
    return frame['text'].tolist()

def expand_proba(p, classes):
    full = np.zeros((len(p), N_CLASSES))
    full[:, classes] = p
    return full

class TfidfFeatures:
    def fit(self, texts):
        self.word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                                    max_features=TFIDF_MAX_FEATURES, strip_accents='unicode')
        self.word.fit(texts)
        self.char = None
        if CFG.TFIDF_CHAR:
            self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True,
                                        max_features=TFIDF_MAX_FEATURES)
            self.char.fit(texts)
        return self

    def transform(self, texts):
        mats = [self.word.transform(texts)] + ([self.char.transform(texts)] if self.char is not None else [])
        return sparse.hstack(mats).tocsr()

class LinearHead:
    def fit(self, X, y):
        if IS_REG:
            self.mu, self.sd = y.mean(), y.std() + 1e-9
            self.est = Ridge(alpha=CFG.BASELINE_ALPHA).fit(X, (y - self.mu) / self.sd)
        elif IS_ML:
            self.est = OneVsRestClassifier(LogisticRegression(C=CFG.BASELINE_C, max_iter=3000,
                                                              class_weight='balanced' if USE_CW else None)).fit(X, y)
        else:
            self.classes, yl = np.unique(y, return_inverse=True)
            self.est = LogisticRegression(C=CFG.BASELINE_C, max_iter=3000,
                                          class_weight='balanced' if USE_CW else None).fit(X, yl)
        return self

    def predict(self, X):
        if IS_REG:
            return self.est.predict(X) * self.sd + self.mu
        if IS_ML:
            return self.est.predict_proba(X)
        return expand_proba(self.est.predict_proba(X), self.classes)

def empty_pred(n):
    return np.full((n,), np.nan) if IS_REG else np.full((n, N_CLASSES), np.nan)

def finish_result(name, oof, hold_p, test_p, fold_scores, t0, extra=None):
    covered = ~np.isnan(oof if IS_REG else oof[:, 0])
    cv = score_all(y_dev[covered], oof[covered])
    res = {'name': name, 'oof': oof, 'covered': covered, 'hold': hold_p, 'test': test_p, 'fold_scores': fold_scores,
           'cv': cv, 'cv_primary': cv[METRIC], 'cv_std': float(np.std(fold_scores)), 'time': time.time() - t0,
           'holdout': score_all(y_hold, hold_p) if HAS_HOLD else {}}
    res.update(extra or {})
    print(f'[{name}] CV {METRIC} = {res["cv_primary"]:.4f} +/- {res["cv_std"]:.4f} on {covered.sum():,} rows'
          + (f' | holdout {res["holdout"][METRIC]:.4f}' if HAS_HOLD else '') + f' | {res["time"]:.0f}s')
    return res

TX_DEV, TX_HOLD = combine(dev), combine(hold) if HAS_HOLD else []
TX_TEST = combine(test) if test is not None else None
t0 = time.time()
TFIDF = TfidfFeatures().fit(TX_DEV)
X_DEV = TFIDF.transform(TX_DEV)
X_HOLD = TFIDF.transform(TX_HOLD) if HAS_HOLD else None
X_TEST = TFIDF.transform(TX_TEST) if TX_TEST is not None else None
print(f'TF-IDF matrix: {X_DEV.shape[0]:,} x {X_DEV.shape[1]:,} ({time.time() - t0:.0f}s)')
oof, fold_scores = empty_pred(len(dev)), []
hold_p = np.zeros(empty_pred(len(hold)).shape)
test_p = np.zeros(empty_pred(len(test)).shape) if test is not None else None
for k, (tr, va) in enumerate(FOLDS):
    m = LinearHead().fit(X_DEV[tr], y_dev[tr])
    oof[va] = m.predict(X_DEV[va])
    fold_scores.append(primary(y_dev[va], oof[va]))
    print(f'  [tfidf] fold {k + 1}/{len(FOLDS)} {METRIC} = {fold_scores[-1]:.4f}')
    if HAS_HOLD:
        hold_p += m.predict(X_HOLD) / len(FOLDS)
    if test is not None:
        test_p += m.predict(X_TEST) / len(FOLDS)
BASE = finish_result('tfidf', oof, hold_p, test_p, fold_scores, t0)
RESULTS = {'tfidf': BASE}

## Baseline Diagnostics

The diagnostics function is shared with the final evaluation. Classification shows a row-normalised confusion matrix and per-class F1. Regression shows predicted against actual values and the residual distribution.

In [ ]:
def plot_diagnostics(y_true, pred, title, stage, subtitle=''):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6) if not IS_REG else (14, 5))
    if IS_REG:
        axes[0].hexbin(y_true, pred, gridsize=40, cmap=CMAP_SEQ, mincnt=1, bins='log', linewidths=0)
        lims = [min(y_true.min(), pred.min()), max(y_true.max(), pred.max())]
        axes[0].plot(lims, lims, color=ACCENT, ls='--')
        axes[0].set_xlabel('actual')
        axes[0].set_ylabel('predicted')
        axes[0].set_title('Predicted vs actual', loc='left')
        sns.histplot(y_true - pred, bins=60, color=PRIMARY, edgecolor='white', kde=True, ax=axes[1])
        axes[1].axvline(0, color=ACCENT, ls='--')
        axes[1].set_title('Residuals (actual - predicted)', loc='left')
    elif IS_ML:
        lab = to_label(pred)
        freq = pd.DataFrame({'actual': y_true.mean(axis=0), 'predicted': lab.mean(axis=0)}, index=CLASS_NAMES).sort_values('actual')
        yy = np.arange(len(freq))
        axes[0].barh(yy - 0.2, freq['actual'], height=0.4, color=PRIMARY, label='actual')
        axes[0].barh(yy + 0.2, freq['predicted'], height=0.4, color=ACCENT, label='predicted')
        axes[0].set_yticks(yy)
        axes[0].set_yticklabels([short(i, 18) for i in freq.index], fontsize=8)
        axes[0].legend(loc='lower right')
        axes[0].set_title('Label frequency: actual vs predicted', loc='left')
        p, r, f, _ = precision_recall_fscore_support(y_true, lab, zero_division=0)
        per = pd.DataFrame({'precision': p, 'recall': r, 'f1': f}, index=CLASS_NAMES).sort_values('f1')
        yy = np.arange(len(per))
        axes[1].barh(yy, per['f1'], color=[ACCENT if v < per['f1'].median() else PRIMARY for v in per['f1']], height=0.6, label='F1')
        axes[1].scatter(per['precision'], yy, marker='|', s=120, color=DARK, label='precision')
        axes[1].scatter(per['recall'], yy, marker='x', s=30, color=PALETTE[2], label='recall')
        axes[1].set_yticks(yy)
        axes[1].set_yticklabels([short(i, 18) for i in per.index], fontsize=8)
        axes[1].set_xlim(0, 1.02)
        axes[1].legend(loc='lower right', fontsize=8)
        axes[1].set_title('Per-label F1, precision, recall', loc='left')
    else:
        lab = to_label(pred)
        keep = np.sort(np.argsort(-np.bincount(y_true, minlength=N_CLASSES))[:15])
        cm = confusion_matrix(y_true, lab, labels=list(range(N_CLASSES)), normalize='true')[np.ix_(keep, keep)]
        names = [short(CLASS_NAMES[i], 14) for i in keep]
        sns.heatmap(cm, cmap=CMAP_SEQ, vmin=0, vmax=1, annot=len(keep) <= 12, fmt='.2f', ax=axes[0],
                    xticklabels=names, yticklabels=names, cbar_kws={'shrink': 0.7})
        axes[0].set_xlabel('predicted')
        axes[0].set_ylabel('actual')
        axes[0].set_title('Row-normalised confusion matrix', loc='left')
        p, r, f, s = precision_recall_fscore_support(y_true, lab, labels=list(range(N_CLASSES)), zero_division=0)
        per = pd.DataFrame({'precision': p, 'recall': r, 'f1': f}, index=CLASS_NAMES).sort_values('f1')
        per = per if len(per) <= 25 else pd.concat([per.head(12), per.tail(12)])
        yy = np.arange(len(per))
        axes[1].barh(yy, per['f1'], color=[ACCENT if v < per['f1'].median() else PRIMARY for v in per['f1']], height=0.6, label='F1')
        axes[1].scatter(per['precision'], yy, marker='|', s=120, color=DARK, label='precision')
        axes[1].scatter(per['recall'], yy, marker='x', s=30, color=PALETTE[2], label='recall')
        axes[1].set_yticks(yy)
        axes[1].set_yticklabels([short(i, 18) for i in per.index], fontsize=8)
        axes[1].set_xlim(0, 1.02)
        axes[1].legend(loc='lower right', fontsize=8)
        axes[1].set_title('Per-class F1, precision, recall', loc='left')
    suptitle(fig, title, subtitle)
    save_fig(fig, title, stage)

cov = BASE['covered']
plot_diagnostics(y_dev[cov], BASE['oof'][cov], 'TF-IDF baseline out-of-fold diagnostics', 'baseline', f'CV {METRIC} = {BASE["cv_primary"]:.4f}')
if HAS_HOLD:
    plot_diagnostics(y_hold, BASE['hold'], 'TF-IDF baseline holdout diagnostics', 'baseline', f'holdout {METRIC} = {BASE["holdout"][METRIC]:.4f}')

## Most Predictive N-grams

A baseline refitted on the full development set exposes the word n-grams with the largest coefficients per class (or the strongest positive and negative words for regression). These are directly quotable evidence for 'what drives each category' in the report.

In [ ]:
BASE_FULL = LinearHead().fit(X_DEV, y_dev)
vocab = TFIDF.word.get_feature_names_out()
n_word = len(vocab)
coef = None if IS_ML else BASE_FULL.est.coef_[:, :n_word] if not IS_REG else BASE_FULL.est.coef_[:n_word][None, :]
rows = []
if IS_ML:
    panels = {CLASS_NAMES[j]: pd.Series(e.coef_[0][:n_word], index=vocab).nlargest(12)
              for j, e in enumerate(BASE_FULL.est.estimators_) if hasattr(e, 'coef_')}
elif IS_REG or N_CLASSES == 2:
    c = coef[0]
    pos_name = 'higher target' if IS_REG else CLASS_NAMES[BASE_FULL.classes[1]]
    neg_name = 'lower target' if IS_REG else CLASS_NAMES[BASE_FULL.classes[0]]
    panels = {pos_name: pd.Series(c, index=vocab).nlargest(15), neg_name: (-pd.Series(c, index=vocab)).nlargest(15)}
else:
    panels = {CLASS_NAMES[cls]: pd.Series(coef[j], index=vocab).nlargest(12) for j, cls in enumerate(BASE_FULL.classes)}
for name, s in panels.items():
    rows += [{'class': name, 'ngram': w, 'weight': v} for w, v in s.items()]
pd.DataFrame(rows).to_csv(CFG.BASELINE_DIR / 'top_ngrams.csv', index=False)

show = list(panels.items())[:12]
ncols = min(4, len(show))
nrows = int(np.ceil(len(show) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.9 * nrows), squeeze=False)
for ax in axes.ravel()[len(show):]:
    ax.set_visible(False)
for i, (ax, (name, s)) in enumerate(zip(axes.ravel(), show)):
    s = s[::-1]
    ax.barh([short(w, 22) for w in s.index], s.values, color=PALETTE[i % 10])
    ax.set_title(short(name, 26), loc='left', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
suptitle(fig, 'TF-IDF baseline: most predictive word n-grams', 'largest logistic / ridge coefficients per class')
save_fig(fig, 'top_ngrams', 'baseline')

## Baseline Export

Baseline metrics, fold scores, out-of-fold and holdout predictions, and a submission file (when a test set is configured) are written to `pipeline-output/baseline/`.

In [ ]:
def prediction_frame(pred, frame=None, truth=None):
    out = pd.DataFrame(index=range(len(pred)))
    if frame is not None and 'id' in frame.columns:
        out[CFG.ID_COL or 'id'] = frame['id'].values
    if truth is not None:
        out['actual'] = ([ml_names(r) for r in truth] if IS_ML else np.array(CLASS_NAMES, dtype=object)[np.asarray(truth).astype(int)]
                         if IS_ORD else truth if IS_REG else LE.inverse_transform(truth))
    if IS_ORD:
        out[LABEL_COL] = pred_labels(pred)
        out['score'] = pred
    elif IS_REG:
        out[LABEL_COL] = pred
    else:
        out[LABEL_COL] = pred_labels(pred)
        for j, c in enumerate(CLASS_NAMES):
            out[f'proba_{c}'] = pred[:, j]
    return out

def follow_sample(pred, frame):
    """Shape test predictions like sample_submission: same id order, column names, and label or probability format."""
    sid, tcols = SAMPLE.columns[0], list(SAMPLE.columns[1:])
    if sid not in frame.columns or not tcols or len(frame) != len(pred):
        print(f'[warn] sample submission id column {sid!r} is not in the test file; using the default submission layout')
        return None
    names = [str(c) for c in CLASS_NAMES] if not IS_REG else []
    if not IS_REG and len(tcols) > 1 and set(map(str, tcols)) <= set(names):
        cols_idx = [names.index(str(c)) for c in tcols]
        ex = SAMPLE[tcols].stack().dropna()
        as_proba = CFG.SUBMISSION_FORMAT == 'proba' or (CFG.SUBMISSION_FORMAT == 'auto' and pd.api.types.is_float_dtype(ex) and not ex.isin([0, 1]).all())
        vals = pd.DataFrame((pred if (as_proba or not IS_ML) else to_label(pred))[:, cols_idx], columns=tcols)
    else:
        ex = SAMPLE[tcols[0]].dropna()
        looks_proba = pd.api.types.is_float_dtype(ex) and not ex.isin([0, 1]).all()
        fmt = CFG.SUBMISSION_FORMAT if CFG.SUBMISSION_FORMAT != 'auto' else 'proba' if looks_proba else 'label'
        if IS_REG and not IS_ORD:
            v = pred
        elif fmt == 'proba' and N_CLASSES == 2:
            v = pred[:, 1]
        else:
            v = pred_labels(pred)
        vals = pd.DataFrame({tcols[0]: v})
    vals.insert(0, sid, frame[sid].values)
    out = SAMPLE[[sid]].merge(vals, on=sid, how='left')
    n_missing = int(out[vals.columns[1:]].isna().any(axis=1).sum())
    if n_missing:
        print(f'[warn] {n_missing:,} sample-submission ids have no prediction')
    return out

def export_result(res, folder):
    cov = res['covered']
    prediction_frame(res['oof'][cov], dev[cov], y_dev[cov]).to_csv(folder / 'oof_predictions.csv', index=False)
    if HAS_HOLD:
        prediction_frame(res['hold'], hold, y_hold).to_csv(folder / 'holdout_predictions.csv', index=False)
    if res['test'] is not None:
        sub = prediction_frame(res['test'], test)
        id_cols = [c for c in (CFG.ID_COL or 'id',) if c in sub.columns]
        sub_out = follow_sample(res['test'], raw_test) if SAMPLE is not None else None
        if sub_out is None:
            fmt = CFG.SUBMISSION_FORMAT
            if fmt == 'auto':
                fmt = 'proba' if not IS_REG and METRIC in ('roc_auc', 'logloss') else 'label'
            if IS_ML and ML_COLUMNS and fmt != 'proba':
                sub_out = sub[id_cols].assign(**{c: to_label(res['test'])[:, j] for j, c in enumerate(CLASS_NAMES)})
            elif fmt == 'proba' and not IS_REG and not IS_ML and N_CLASSES == 2:
                sub_out = sub[id_cols].assign(**{LABEL_COL: res['test'][:, 1]})
            elif fmt == 'proba' and not IS_REG:
                sub_out = sub[id_cols + [f'proba_{c}' for c in CLASS_NAMES]].rename(columns=lambda c: c.removeprefix('proba_'))
            else:
                sub_out = sub[id_cols + [LABEL_COL]]
        sub_out.to_csv(folder / 'submission.csv', index=False)
        sub.to_csv(folder / 'test_predictions_full.csv', index=False)

export_result(BASE, CFG.BASELINE_DIR)
pd.DataFrame({'fold': range(1, len(FOLDS) + 1), METRIC: BASE['fold_scores']}).to_csv(CFG.BASELINE_DIR / 'cv_scores.csv', index=False)
to_json({'model': 'TF-IDF (word 1-2 + char_wb 2-5) + ' + ('Ridge' if IS_REG else 'LogisticRegression'),
         'tfidf_max_features': TFIDF_MAX_FEATURES, 'metric': METRIC, 'cv': BASE['cv'], 'cv_fold_scores': BASE['fold_scores'],
         'holdout': BASE['holdout']}, CFG.BASELINE_DIR / 'metrics.json')
print('Baseline files:', sorted(p.name for p in CFG.BASELINE_DIR.iterdir()))

#### Insights

> *Fill in after running.* What score does the baseline reach, and which classes does it confuse? Do the top n-grams make domain sense? A strong baseline means the task is largely lexical, and transformers must beat it clearly to justify their cost.

---

# Transformer Setup <a name="7"></a>

---

Each requested model key is resolved to a checkpoint (unknown keys are treated as Hugging Face ids). Its own tokenizer then measures the real token length of the texts, which verifies that the EDA-derived `MAX_LENGTH` covers most documents for that vocabulary. Indonesian text, for example, needs more tokens per word in an English tokenizer than in IndoBERT's.

In [ ]:
TRANSFORMERS_OK = bool(AVAILABLE['torch'])
ACTIVE = []
if not TRANSFORMERS_OK:
    note('PyTorch / Transformers are not installed, so only the TF-IDF baseline is used.')
elif not MODELS:
    note('MODELS is empty, so only the TF-IDF baseline is used.')
else:
    ACTIVE = list(dict.fromkeys(MODELS))
    if DEVICE == 'cpu':
        print('[warn] no GPU detected: fine-tuning will be very slow; consider MODELS = [] or a tiny model')
CHECKPOINT = {k: CFG.MODEL_REGISTRY.get(k, k) for k in ACTIVE}

def hp(key):
    base = {'lr': CFG.LR, 'batch_size': CFG.BATCH_SIZE, 'epochs': CFG.EPOCHS, 'max_length': MAX_LENGTH,
            'grad_accum': CFG.GRAD_ACCUM, 'dropout': CFG.DROPOUT, 'weight_decay': CFG.WEIGHT_DECAY}
    base.update(CFG.MODEL_OVERRIDES.get(key, {}))
    return base

TOKENIZERS, LEN_STATS = {}, []
for key in ACTIVE:
    try:
        try:
            TOKENIZERS[key] = AutoTokenizer.from_pretrained(CHECKPOINT[key])
        except Exception:
            TOKENIZERS[key] = AutoTokenizer.from_pretrained(CHECKPOINT[key], use_fast=False)
    except Exception as e:
        print(f'[warn] cannot load tokenizer for {key} ({CHECKPOINT[key]}): {str(e).splitlines()[0][:150]}')
ACTIVE = [k for k in ACTIVE if k in TOKENIZERS]
print('Active transformer models:', {k: CHECKPOINT[k] for k in ACTIVE})

## Token Length Coverage

Coverage is the share of documents that fit in `MAX_LENGTH` tokens without truncation. With `TRUNCATION = 'head_tail'`, documents that do not fit keep their first `HEAD_RATIO` of the budget and the end of the text, which is where reviews and essays usually state their conclusion (Sun et al., 2019).

In [ ]:
if ACTIVE:
    samp = random.Random(CFG.SEED).sample(TX_DEV, min(1500, len(TX_DEV)))
    fig, axes = plt.subplots(1, len(ACTIVE), figsize=(6 * len(ACTIVE), 4.2), squeeze=False)
    for ax, key in zip(axes[0], ACTIVE):
        lens = np.array([len(x) for x in TOKENIZERS[key](samp, add_special_tokens=True, truncation=False)['input_ids']])
        L = hp(key)['max_length']
        cov = (lens <= L).mean()
        LEN_STATS.append({'model': key, 'median_tokens': np.median(lens), 'p95_tokens': np.percentile(lens, 95),
                          'max_length': L, 'coverage_%': cov * 100})
        sns.histplot(np.clip(lens, 0, np.percentile(lens, 99)), bins=50, color=PRIMARY, edgecolor='white', ax=ax)
        ax.axvline(L, color=ACCENT, ls='--', lw=2, label=f'max_length {L}')
        ax.legend()
        ax.set_title(f'{key}: {cov:.0%} of docs fit', loc='left')
        ax.set_xlabel('tokens per document')
    suptitle(fig, 'Token length per tokenizer', f'{len(samp):,} sampled documents; truncation: {CFG.TRUNCATION}')
    save_fig(fig, 'token_lengths', 'final')
    display(pd.DataFrame(LEN_STATS).set_index('model').round(1))

## Encoding

Texts are tokenised once per model. For head-plus-tail truncation the special tokens of each tokenizer are detected empirically, so the same code works for BERT, RoBERTa, DeBERTa, and ModernBERT style tokenizers. Sentence pairs use the tokenizer's native pair truncation.

In [ ]:
def special_wrap(tok):
    core = tok('hello', add_special_tokens=False)['input_ids']
    full = tok('hello', add_special_tokens=True)['input_ids']
    for s in range(len(full) - len(core) + 1):
        if full[s:s + len(core)] == core:
            return full[:s], full[s + len(core):]
    return full[:1], full[-1:]

def encode(key, frame):
    tok, L = TOKENIZERS[key], hp(key)['max_length']
    texts = frame['text'].tolist()
    if CFG.TEXT_PAIR_COL:
        enc = tok(texts, frame['text_pair'].tolist(), truncation='longest_first', max_length=L)
        return [{k: enc[k][i] for k in enc.keys()} for i in range(len(texts))]
    if CFG.TRUNCATION != 'head_tail':
        enc = tok(texts, truncation=True, max_length=L)
        return [{k: enc[k][i] for k in enc.keys()} for i in range(len(texts))]
    pre, post = special_wrap(tok)
    budget = L - len(pre) - len(post)
    out = []
    for ids in tok(texts, add_special_tokens=False, truncation=False)['input_ids']:
        if len(ids) > budget:
            h = int(budget * CFG.HEAD_RATIO)
            ids = ids[:h] + ids[len(ids) - (budget - h):]
        ids = pre + ids + post
        out.append({'input_ids': ids, 'attention_mask': [1] * len(ids)})
    return out

ENC = {}
for key in ACTIVE:
    t0 = time.time()
    ENC[key] = {'dev': encode(key, dev), 'hold': encode(key, hold) if HAS_HOLD else [],
                'test': encode(key, test) if test is not None else None}
    print(f'{key}: encoded {len(ENC[key]["dev"]):,} dev texts in {time.time() - t0:.0f}s, '
          f'example length {len(ENC[key]["dev"][0]["input_ids"])}')

---

# Transformer Fine-Tuning <a name="8"></a>

---

The architecture is the pretrained encoder, mean pooling over non-padding tokens, dropout, and one linear layer (one output per class, or one output for regression). Mean pooling uses every token and is more robust across model families than the first token alone, especially for encoders without a next-sentence pretraining objective such as RoBERTa or ModernBERT.

$$h = \frac{\sum_{t} m_t \, H_t}{\sum_t m_t}, \qquad \hat{y} = W \, \text{Dropout}(h) + b$$

where $H_t$ is the last-layer hidden state of token $t$ and $m_t$ is its attention mask.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, enc, y=None):
        self.enc, self.y = enc, y

    def __len__(self):
        return len(self.enc)

    def __getitem__(self, i):
        item = dict(self.enc[i])
        if self.y is not None:
            item['labels'] = self.y[i]
        return item


class TextModel(nn.Module):
    def __init__(self, checkpoint, n_out, dropout):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(checkpoint)
        if CFG.GRAD_CHECKPOINT:
            self.backbone.gradient_checkpointing_enable()
        self.accepts_tt = 'token_type_ids' in inspect.signature(self.backbone.forward).parameters
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.backbone.config.hidden_size, n_out)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {'token_type_ids': token_type_ids} if (token_type_ids is not None and self.accepts_tt) else {}
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kw).last_hidden_state
        if CFG.POOLING == 'cls':
            pooled = hidden[:, 0]
        else:
            m = attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * m).sum(1) / m.sum(1).clamp(min=1e-6)
        return self.head(self.dropout(pooled))


def amp_dtype():
    if DEVICE != 'cuda' or CFG.MIXED_PRECISION == 'no':
        return None
    if CFG.MIXED_PRECISION == 'bf16' or (CFG.MIXED_PRECISION == 'auto' and torch.cuda.is_bf16_supported()):
        return torch.bfloat16
    return torch.float16

## Training Loop

Training uses AdamW with a higher learning rate for the freshly initialised head than for the pretrained encoder, a cosine schedule with linear warmup, gradient clipping at 1.0, optional gradient accumulation, and mixed precision on CUDA (bfloat16 when supported, float16 with loss scaling otherwise). The validation fold is scored `EVALS_PER_EPOCH` times per epoch, the best checkpoint is kept in memory, and training stops after `PATIENCE` evaluations without improvement. Regression targets are standardised per fold and mapped back for scoring. Inference sorts texts by length to minimise padding.

In [ ]:
def better(a, b):
    return a > b if GREATER_IS_BETTER[METRIC] else a < b

@torch.no_grad()
def predict_encoded(model, enc, collate, bs, mu=0.0, sd=1.0):
    if not enc:
        return empty_pred(0)
    model.eval()
    order = np.argsort([len(e['input_ids']) for e in enc])
    loader = DataLoader(TextDataset([enc[i] for i in order]), batch_size=bs, shuffle=False, collate_fn=collate)
    dtype, outs = amp_dtype(), []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=dtype or torch.float16, enabled=dtype is not None):
            outs.append(model(**batch).float().cpu())
    logits = torch.cat(outs).numpy()
    pred = (logits[:, 0] * sd + mu if IS_REG else torch.sigmoid(torch.from_numpy(logits)).numpy() if IS_ML
            else torch.softmax(torch.from_numpy(logits), dim=-1).numpy())
    result = np.empty_like(pred)
    result[order] = pred
    return result

def run_fold(key, fold, tr, va):
    p, tok = hp(key), TOKENIZERS[key]
    collate = DataCollatorWithPadding(tok)
    enc_dev = ENC[key]['dev']
    ytr, yva = y_dev[tr], y_dev[va]
    mu, sd = (float(ytr.mean()), float(ytr.std() + 1e-9)) if IS_REG else (0.0, 1.0)
    ytr_t = ((ytr - mu) / sd).astype(np.float32) if IS_REG else ytr.astype(np.float32) if IS_ML else ytr.astype(np.int64)
    loader = DataLoader(TextDataset([enc_dev[i] for i in tr], ytr_t.tolist()), batch_size=p['batch_size'],
                        shuffle=True, collate_fn=collate, drop_last=len(tr) > p['batch_size'])
    enc_va = [enc_dev[i] for i in va]

    core = TextModel(CHECKPOINT[key], 1 if IS_REG else N_CLASSES, p['dropout']).to(DEVICE)
    model = nn.DataParallel(core) if DEVICE == 'cuda' and torch.cuda.device_count() > 1 else core
    opt = torch.optim.AdamW([{'params': core.backbone.parameters(), 'lr': p['lr']},
                             {'params': core.head.parameters(), 'lr': p['lr'] * CFG.HEAD_LR_MULT}],
                            weight_decay=p['weight_decay'])
    total = math.ceil(len(loader) / p['grad_accum']) * p['epochs']
    sched = get_cosine_schedule_with_warmup(opt, int(CFG.WARMUP_RATIO * total), total)
    dtype = amp_dtype()
    scaler = torch.amp.GradScaler(enabled=dtype == torch.float16)
    if IS_REG:
        loss_fn = nn.MSELoss()
    elif IS_ML:
        pos = np.clip(ytr.mean(axis=0), 1e-3, 1)
        pw = torch.tensor((1 - pos) / pos, dtype=torch.float32, device=DEVICE) if USE_CW else None
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        counts = np.bincount(ytr, minlength=N_CLASSES).astype(float)
        w = len(ytr) / (np.count_nonzero(counts) * np.where(counts == 0, 1, counts))
        weight = torch.tensor(w, dtype=torch.float32, device=DEVICE) if USE_CW else None
        loss_fn = nn.CrossEntropyLoss(weight=weight, label_smoothing=CFG.LABEL_SMOOTHING)

    eval_every = max(1, len(loader) // CFG.EVALS_PER_EPOCH)
    best, best_state, bad, history, step, t0 = None, None, 0, [], 0, time.time()
    for epoch in range(p['epochs']):
        model.train()
        running, n_batches = 0.0, 0
        for i, batch in enumerate(loader):
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=dtype or torch.float16, enabled=dtype is not None):
                out = model(**batch).float()
            loss = loss_fn(out[:, 0], labels.float()) if IS_REG else loss_fn(out, labels.float()) if IS_ML else loss_fn(out, labels.long())
            scaler.scale(loss / p['grad_accum']).backward()
            running, n_batches = running + loss.item(), n_batches + 1
            if (i + 1) % p['grad_accum'] == 0 or i + 1 == len(loader):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(core.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                step += 1
            if (i + 1) % eval_every == 0 or i + 1 == len(loader):
                pv = predict_encoded(core, enc_va, collate, p['batch_size'] * 2, mu, sd)
                score = primary(yva, pv)
                history.append({'model': key, 'fold': fold, 'epoch': epoch + (i + 1) / len(loader), 'train_loss': running / n_batches, METRIC: score})
                improved = best is None or better(score, best)
                if improved:
                    best, bad = score, 0
                    best_state = {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}
                else:
                    bad += 1
                print(f'  [{key} f{fold}] epoch {epoch + (i + 1) / len(loader):.2f} loss {running / n_batches:.4f} '
                      f'{METRIC} {score:.4f}{" *" if improved else ""} ({time.time() - t0:.0f}s)')
                running, n_batches = 0.0, 0
                model.train()
                if bad >= CFG.PATIENCE:
                    break
        if bad >= CFG.PATIENCE:
            print(f'  [{key} f{fold}] early stopping')
            break
    core.load_state_dict(best_state)
    if CFG.SAVE_WEIGHTS:
        torch.save(best_state, CFG.FINAL_DIR / 'models' / f'{slug(key)}_fold{fold}.pt')
    bs = p['batch_size'] * 2
    preds = {'va': predict_encoded(core, enc_va, collate, bs, mu, sd),
             'hold': predict_encoded(core, ENC[key]['hold'], collate, bs, mu, sd) if HAS_HOLD else None,
             'test': predict_encoded(core, ENC[key]['test'], collate, bs, mu, sd) if ENC[key]['test'] is not None else None}
    return core, preds, history, (mu, sd)

## Cross-Validated Fine-Tuning

Each active model is fine-tuned on the folds listed in `TRAIN_FOLDS` (all folds is the most reliable, `[0]` is the fast single-split option). Out-of-fold predictions exist only for the trained folds, and holdout and test predictions are averaged over the trained fold models. The first fold's model is kept in memory for interpretation, and all others are freed to save GPU memory.

In [ ]:
TRAIN_FOLDS = list(range(CFG.N_FOLDS)) if CFG.TRAIN_FOLDS in ('all', None) else list(CFG.TRAIN_FOLDS)
HISTORY, KEPT = [], {}
for key in ACTIVE:
    oof = empty_pred(len(dev))
    hold_p = np.zeros(empty_pred(len(hold)).shape)
    test_p = np.zeros(empty_pred(len(test)).shape) if test is not None else None
    fold_scores, t0 = [], time.time()
    print(f'=== {key} ({CHECKPOINT[key]}) | hyperparameters {hp(key)}')
    for fold in TRAIN_FOLDS:
        tr, va = FOLDS[fold]
        seed_everything(CFG.SEED + fold)
        core, preds, hist, scale = run_fold(key, fold, tr, va)
        HISTORY += hist
        oof[va] = preds['va']
        fold_scores.append(primary(y_dev[va], preds['va']))
        if HAS_HOLD:
            hold_p += preds['hold'] / len(TRAIN_FOLDS)
        if test is not None:
            test_p += preds['test'] / len(TRAIN_FOLDS)
        if key not in KEPT:
            KEPT[key] = (core.cpu(), scale)
        else:
            del core
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    RESULTS[key] = finish_result(key, oof, hold_p, test_p, fold_scores, t0)
    cov = RESULTS[key]['covered']
    prediction_frame(oof[cov], dev[cov], y_dev[cov]).to_csv(CFG.FINAL_DIR / 'oof' / f'{slug(key)}.csv', index=False)
if HISTORY:
    pd.DataFrame(HISTORY).to_csv(CFG.FINAL_DIR / 'training_history.csv', index=False)

## Learning Curves

Training loss and the validation metric are plotted against epochs for every model and fold. A validation curve that peaks early and then degrades signals overfitting, which calls for fewer epochs, a lower learning rate, or more dropout. A curve still rising at the end calls for more epochs.

In [ ]:
if HISTORY:
    h = pd.DataFrame(HISTORY)
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
    for i, ((key, fold), g) in enumerate(h.groupby(['model', 'fold'])):
        col = PALETTE[ACTIVE.index(key) % 10]
        axes[0].plot(g['epoch'], g['train_loss'], color=col, marker='o', ms=3, lw=1.6, alpha=0.85, label=f'{key} f{fold}')
        axes[1].plot(g['epoch'], g[METRIC], color=col, marker='o', ms=3, lw=1.6, alpha=0.85, label=f'{key} f{fold}')
    axes[1].axhline(BASE['cv_primary'], color=MUTED, ls='--', lw=1.2, label='TF-IDF baseline (CV)')
    axes[0].set_title('Training loss', loc='left')
    axes[1].set_title(f'Validation {METRIC}', loc='left')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.legend(fontsize=8)
    suptitle(fig, 'Learning curves', 'one line per model and fold; dashed = baseline')
    save_fig(fig, 'learning_curves', 'final')
else:
    note('No transformer was trained.')

#### Insights

> *Fill in after running.* How much does each transformer improve on the TF-IDF baseline, and is the gain consistent across folds? Do the learning curves show under- or over-fitting, and what change to epochs, learning rate, or `MAX_LENGTH` follows from that?

---

# Ensemble and Model Selection <a name="9"></a>

---

TF-IDF models and transformers make different errors (exact lexical cues versus contextual meaning), so blending them is usually the single cheapest improvement. Non-negative weights summing to one are fitted on the out-of-fold rows shared by all candidates, minimising log loss (classification) or RMSE (regression). The final model is the candidate with the best cross-validated score on those shared rows. The holdout score is reported but not used to choose.

In [ ]:
def blend(arrs, w):
    return sum(wi * a for wi, a in zip(w, arrs))

WEIGHTS = None
cands = list(RESULTS)
SHARED = np.logical_and.reduce([RESULTS[k]['covered'] for k in cands])
if CFG.ENSEMBLE and len(cands) >= 2:
    oofs = [RESULTS[k]['oof'][SHARED] for k in cands]
    yc = y_dev[SHARED]

    def blend_loss(w):
        p = blend(oofs, w)
        if IS_REG:
            return np.sqrt(np.mean((yc - p) ** 2))
        if IS_ML:
            q = np.clip(p, 1e-7, 1 - 1e-7)
            return -np.mean(yc * np.log(q) + (1 - yc) * np.log(1 - q))
        return log_loss(yc, np.clip(p, 1e-7, 1), labels=list(range(N_CLASSES)))

    n = len(cands)
    opt = minimize(blend_loss, np.full(n, 1 / n), method='SLSQP', bounds=[(0, 1)] * n,
                   constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
    WEIGHTS = {k: round(float(w), 4) for k, w in zip(cands, opt.x)}
    print('Ensemble weights:', WEIGHTS)
    oof = empty_pred(len(dev))
    oof[SHARED] = blend(oofs, opt.x)
    fs = [primary(y_dev[va], oof[va]) for f, (_, va) in enumerate(FOLDS) if SHARED[va].all()]
    RESULTS['ensemble'] = finish_result('ensemble', oof, blend([RESULTS[k]['hold'] for k in cands], opt.x),
                                        blend([RESULTS[k]['test'] for k in cands], opt.x) if test is not None else None, fs, time.time())
    to_json(WEIGHTS, CFG.FINAL_DIR / 'ensemble_weights.json')
else:
    note('Ensembling needs ENSEMBLE = True and at least two models (TF-IDF plus one transformer).')

rows = []
for name, r in RESULTS.items():
    shared_score = primary(y_dev[SHARED], r['oof'][SHARED]) if r['covered'][SHARED].all() else np.nan
    row = {'model': name, f'cv_{METRIC}_shared': shared_score, f'cv_{METRIC}_all': r['cv_primary'], 'cv_std': r['cv_std'],
           'oof_rows': int(r['covered'].sum()), f'holdout_{METRIC}': r['holdout'].get(METRIC, np.nan), 'seconds': r['time']}
    row.update({f'holdout_{k}': v for k, v in r['holdout'].items() if k != METRIC})
    rows.append(row)
LEADERBOARD = pd.DataFrame(rows).set_index('model').sort_values(f'cv_{METRIC}_shared', ascending=not GREATER_IS_BETTER[METRIC])
BEST_NAME = LEADERBOARD.index[0]
BEST = RESULTS[BEST_NAME]
LEADERBOARD.to_csv(CFG.FINAL_DIR / 'leaderboard.csv')
display(LEADERBOARD.round(4))
print(f'Selected: {BEST_NAME} (CV {METRIC} on shared rows = {LEADERBOARD.loc[BEST_NAME, f"cv_{METRIC}_shared"]:.4f})')

In [ ]:
lb = LEADERBOARD.iloc[::-1]
fig, axes = plt.subplots(1, 2 if WEIGHTS else 1, figsize=(17 if WEIGHTS else 11, 0.6 * len(lb) + 2.2), squeeze=False)
ax = axes[0, 0]
colors = [ACCENT if m == BEST_NAME else MUTED if m == 'tfidf' else PRIMARY for m in lb.index]
ax.barh(lb.index, lb[f'cv_{METRIC}_shared'], color=colors)
if HAS_HOLD:
    ax.scatter(lb[f'holdout_{METRIC}'], lb.index, marker='D', color=DARK, s=40, zorder=3, label='holdout')
    ax.legend(loc='lower right')
vals = pd.concat([lb[f'cv_{METRIC}_shared']] + ([lb[f'holdout_{METRIC}']] if HAS_HOLD else [])).dropna()
pad = max((vals.max() - vals.min()) * 0.3, abs(vals.max()) * 0.02, 1e-6)
ax.set_xlim(vals.min() - pad, vals.max() + 2 * pad)
for i, v in enumerate(lb[f'cv_{METRIC}_shared']):
    ax.text(v, i, f'  {v:.4f}', va='center', fontsize=8, color='#4A5568')
ax.set_xlabel(METRIC)
ax.set_title('CV on shared out-of-fold rows (bars) and holdout (diamonds)', loc='left')
if WEIGHTS:
    w = pd.Series(WEIGHTS).sort_values()
    axes[0, 1].barh(w.index, w.values, color=PALETTE[2])
    bar_labels(axes[0, 1], '{:.2f}')
    axes[0, 1].set_title('Ensemble weights', loc='left')
suptitle(fig, 'Model leaderboard', f'selected: {BEST_NAME} (orange), TF-IDF baseline in grey')
save_fig(fig, 'leaderboard', 'final')

#### Insights

> *Fill in after running.* Which model wins, by how much over TF-IDF, and do CV and holdout agree? How is the ensemble weight split between lexical (TF-IDF) and contextual (transformer) models?

---

# Final Evaluation on Holdout <a name="10"></a>

---

When the metric depends on hard labels (F1, accuracy, QWK), the label rule is first tuned on out-of-fold rows only: a threshold for binary tasks, one scale per class for multiclass, the cut-points between classes for ordinal targets, and one threshold per label for multi-label. The selected model is then scored once on the untouched holdout. A bootstrap over holdout rows gives a 95% confidence interval for the score and for the gain over the TF-IDF baseline, which is the statistical evidence that the extra complexity pays off.

In [ ]:
LABEL_METRICS = {'f1': lambda y, l: f1_score(y, l), 'f1_macro': lambda y, l: f1_score(y, l, average='macro', zero_division=0),
                 'f1_weighted': lambda y, l: f1_score(y, l, average='weighted'), 'accuracy': accuracy_score,
                 'balanced_accuracy': balanced_accuracy_score, 'qwk': lambda y, l: cohen_kappa_score(y, l, weights='quadratic'),
                 'mae_class': lambda y, l: -np.mean(np.abs(y - l)),
                 'f1_micro': lambda y, l: f1_score(y, l, average='micro', zero_division=0),
                 'f1_samples': lambda y, l: f1_score(y, l, average='samples', zero_division=1)}

def tune_decision(y, p):
    """Tune the label rule on out-of-fold predictions only."""
    if not CFG.TUNE_DECISION or METRIC not in LABEL_METRICS or (IS_REG and not IS_ORD):
        return None
    fn = LABEL_METRICS[METRIC]
    if IS_ORD:
        y = np.asarray(y).astype(int)
        current = lambda: fn(y, to_ordinal(p))
    else:
        current = lambda: fn(y, to_label(p))
    start = current()
    if IS_ORD:
        cuts = np.arange(N_CLASSES - 1) + 0.5
        CUTS[0] = cuts.copy()
        for _ in range(3):
            for i in range(len(cuts)):
                lo = cuts[i - 1] + 1e-3 if i else min(p.min(), cuts[0] - 1)
                hi = cuts[i + 1] - 1e-3 if i < len(cuts) - 1 else max(p.max(), cuts[-1] + 1)
                best_c, best_s = cuts[i], current()
                for c in np.linspace(lo, hi, 41):
                    cuts[i] = c
                    CUTS[0] = cuts
                    sc = current()
                    if sc > best_s:
                        best_c, best_s = c, sc
                cuts[i] = best_c
                CUTS[0] = cuts.copy()
        rule = 'cut-points ' + ', '.join(f'{c:.2f}' for c in CUTS[0])
    elif IS_ML:
        th = np.full(N_CLASSES, 0.5)
        ML_THRESH[0] = th
        for _ in range(2):
            for k in range(N_CLASSES):
                best_t, best_s = th[k], current()
                for t in np.linspace(0.05, 0.95, 37):
                    th[k] = t
                    ML_THRESH[0] = th
                    sc = current()
                    if sc > best_s:
                        best_t, best_s = t, sc
                th[k] = best_t
                ML_THRESH[0] = th.copy()
        rule = 'label thresholds ' + ', '.join(f'{CLASS_NAMES[k]} {t:.2f}' for k, t in enumerate(ML_THRESH[0][:8]))
    elif N_CLASSES == 2:
        grid = np.linspace(0.05, 0.95, 91)
        scores = []
        for t in grid:
            THRESHOLD[0] = t
            scores.append(current())
        THRESHOLD[0] = float(grid[int(np.argmax(scores))])
        rule = f'threshold {THRESHOLD[0]:.2f}'
    else:
        scale = np.ones(N_CLASSES)
        grid = np.exp(np.linspace(-1.5, 1.5, 25))
        for _ in range(2):
            for k in range(N_CLASSES):
                best_v, best_s = scale[k], current()
                for v in grid:
                    scale[k] = v
                    CLASS_SCALE[0] = scale
                    sc = current()
                    if sc > best_s:
                        best_v, best_s = v, sc
                scale[k] = best_v
                CLASS_SCALE[0] = scale.copy()
        rule = 'class scales ' + ', '.join(f'{v:.2f}' for v in CLASS_SCALE[0][:8]) + (' ...' if N_CLASSES > 8 else '')
    end = current()
    if METRIC == 'mae_class':
        start, end = -start, -end
    print(f'Tuned decision rule on out-of-fold rows ({rule}): {METRIC} {start:.4f} -> {end:.4f}')
    return rule

DECISION_RULE = tune_decision(y_dev[BEST['covered']], BEST['oof'][BEST['covered']])
if HAS_HOLD:
    EVAL_NAME, EVAL_Y, EVAL_FRAME = 'holdout', y_hold, hold
    EVAL_P = {k: RESULTS[k]['hold'] for k in RESULTS}
else:
    note('HOLDOUT_SIZE is 0, so the final evaluation uses shared out-of-fold rows.')
    EVAL_NAME, EVAL_Y, EVAL_FRAME = 'out-of-fold', y_dev[SHARED], dev[SHARED].reset_index(drop=True)
    EVAL_P = {k: RESULTS[k]['oof'][SHARED] for k in RESULTS}

final_table = pd.DataFrame({'tfidf': score_all(EVAL_Y, EVAL_P['tfidf']), BEST_NAME: score_all(EVAL_Y, EVAL_P[BEST_NAME])})
sign = 1 if GREATER_IS_BETTER[METRIC] else -1
rng = np.random.RandomState(CFG.SEED)
boot, gain = [], []
for _ in range(300):
    b = rng.randint(0, len(EVAL_Y), len(EVAL_Y))
    sb = primary(EVAL_Y[b], EVAL_P[BEST_NAME][b])
    boot.append(sb)
    gain.append(sign * (sb - primary(EVAL_Y[b], EVAL_P['tfidf'][b])))
ci, ci_gain = np.nanpercentile(boot, [2.5, 97.5]), np.nanpercentile(gain, [2.5, 97.5])
FINAL = {'model': BEST_NAME, 'evaluated_on': EVAL_NAME, 'metric': METRIC, 'decision_rule': DECISION_RULE, 'scores': final_table[BEST_NAME].to_dict(),
         'baseline_scores': final_table['tfidf'].to_dict(), 'ci95': ci.tolist(), 'gain_vs_baseline_ci95': ci_gain.tolist()}
display(final_table.round(4))
print(f'{BEST_NAME} {EVAL_NAME} {METRIC} = {final_table.loc[METRIC, BEST_NAME]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f})')
if BEST_NAME != 'tfidf':
    print(f'Gain over TF-IDF: 95% CI {ci_gain[0]:+.4f} to {ci_gain[1]:+.4f} ({"significant" if ci_gain[0] > 0 else "not significant"})')
final_table.to_csv(CFG.FINAL_DIR / 'final_metrics.csv')
to_json(FINAL, CFG.FINAL_DIR / 'final_metrics.json')
plot_diagnostics(EVAL_Y, EVAL_P[BEST_NAME], f'Final model ({BEST_NAME}) {EVAL_NAME} diagnostics', 'final',
                 f'{METRIC} = {final_table.loc[METRIC, BEST_NAME]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f}) vs TF-IDF {final_table.loc[METRIC, "tfidf"]:.4f}')

#### Insights

> *Fill in after running.* Quote the final score with its confidence interval and the gain over TF-IDF. Which classes remain weak, and is that explained by size, label noise, or vocabulary overlap found in the EDA?

---

# Model Interpretation <a name="11"></a>

---

Transformers are explained here with word-level occlusion: each word of a document is removed in turn and the drop in the predicted probability of the predicted class (or the change in the predicted value) is that word's importance. It is model-agnostic, needs no extra packages, and produces highlight maps that are easy to read on a slide. Red words support the prediction and blue words oppose it. Four holdout examples are explained: two confident correct predictions and two confident errors (or the best and worst regression predictions).

In [ ]:
TF_KEYS = [k for k in KEPT]
INTERP = max(TF_KEYS, key=lambda k: sign * RESULTS[k]['cv_primary']) if TF_KEYS else None
OCC_MAX_WORDS = 80

def occlusion(key, text, pair=None):
    core, (mu, sd) = KEPT[key]
    core.to(DEVICE)
    words = text.split()[:OCC_MAX_WORDS]
    variants = [' '.join(words)] + [' '.join(words[:i] + words[i + 1:]) for i in range(len(words))]
    frame = pd.DataFrame({'text': variants})
    if CFG.TEXT_PAIR_COL:
        frame['text_pair'] = pair
    preds = predict_encoded(core, encode(key, frame), DataCollatorWithPadding(TOKENIZERS[key]), 32, mu, sd)
    if IS_REG:
        return words, preds[0] - preds[1:], preds[0], None
    target = int(preds[0].argmax())
    return words, preds[0][target] - preds[1:, target], preds[0][target], target

def highlight(words, imp):
    m = max(np.abs(imp).max(), 1e-9)
    spans = []
    for w, v in zip(words, imp):
        a = min(abs(v) / m, 1.0) * 0.85
        rgb = '238,108,77' if v > 0 else '61,90,128'
        spans.append(f'<span style="background: rgba({rgb},{a:.2f}); padding: 1px 2px; border-radius: 3px">{html.escape(w)}</span>')
    return ' '.join(spans)

if INTERP is None:
    note('No transformer was trained, so interpretation relies on the TF-IDF n-grams in section 6.')
else:
    p_eval = EVAL_P[INTERP]
    if IS_REG:
        err = np.abs(EVAL_Y - p_eval)
        picks = list(np.argsort(err)[:2]) + list(np.argsort(-err)[:2])
    elif IS_ML:
        nerr, conf = (to_label(p_eval) != EVAL_Y).sum(axis=1), p_eval.max(axis=1)
        picks = list(np.lexsort((-conf, nerr))[:2]) + list(np.argsort(-nerr, kind='stable')[:2])
    else:
        conf, lab = p_eval.max(axis=1), to_label(p_eval)
        right, wrong = np.where(lab == EVAL_Y)[0], np.where(lab != EVAL_Y)[0]
        picks = list(right[np.argsort(-conf[right])][:2]) + list(wrong[np.argsort(-conf[wrong])][:2])
    blocks, panels = [], []
    for i in picks:
        row = EVAL_FRAME.iloc[i]
        words, imp, base, target = occlusion(INTERP, row['text'], row.get('text_pair'))
        truth = ml_names(EVAL_Y[i]) if IS_ML else CLASS_NAMES[int(EVAL_Y[i])] if IS_ORD else EVAL_Y[i] if IS_REG else CLASS_NAMES[EVAL_Y[i]]
        pred_txt = f'{base:.3f}' if IS_REG else f'{CLASS_NAMES[target]} (p={base:.2f})'
        head = f'<b>actual:</b> {html.escape(str(truth))} &nbsp; <b>predicted:</b> {html.escape(pred_txt)}'
        blocks.append(f'<div style="margin:10px 0;padding:10px;border:1px solid #E2E8F0;border-radius:6px;font-family:sans-serif;line-height:1.9">{head}<br>{highlight(words, imp)}</div>')
        panels.append((head.replace('<b>', '').replace('</b>', '').replace('&nbsp;', ' '), pd.Series(imp, index=[f'{w} ({j})' for j, w in enumerate(words)])))
    page = f'<h3>Word occlusion: {INTERP}</h3>' + ''.join(blocks)
    (CFG.FINAL_DIR / 'occlusion_examples.html').write_text(page, encoding='utf-8')
    display(HTML(page))

    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5), squeeze=False)
    for ax, (title, s) in zip(axes[0], panels):
        top = s.reindex(s.abs().sort_values(ascending=False).index[:12])[::-1]
        ax.barh([short(t, 20) for t in top.index], top.values, color=[ACCENT if v > 0 else PRIMARY for v in top.values])
        ax.axvline(0, color=DARK, lw=0.8)
        ax.set_title(short(title, 48), loc='left', fontsize=9)
        ax.tick_params(axis='y', labelsize=8)
    suptitle(fig, f'Most influential words per example ({INTERP})', 'orange = supports the prediction, blue = opposes it')
    save_fig(fig, 'occlusion_words', 'final')

#### Insights

> *Fill in after running.* Does the model rely on sensible words, or on artefacts such as names, templates, or headers? Errors driven by artefacts point to a cleaning fix in `REMOVE_PATTERNS`.

---

# Error Analysis <a name="12"></a>

---

Errors are broken down by document length (quintiles of word count), which shows whether truncation hurts long documents, and by class or target range. The most confident errors are listed for manual inspection, since in text tasks they are often label noise rather than model mistakes.

In [ ]:
p_best = EVAL_P[BEST_NAME]
n_words = EVAL_FRAME['text'].str.split().str.len().values
bucket = pd.qcut(pd.Series(n_words).rank(method='first'), 5, labels=['Q1 shortest', 'Q2', 'Q3', 'Q4', 'Q5 longest'])
edges = pd.Series(n_words).groupby(bucket.values, observed=True).agg(['min', 'max'])
if IS_REG:
    err = np.abs(EVAL_Y - p_best)
    by_len = pd.Series(err).groupby(bucket.values, observed=True).mean().rename('mae')
    worst_idx = np.argsort(-err)[:15]
    worst = EVAL_FRAME.iloc[worst_idx][['text']].assign(actual=EVAL_Y[worst_idx], predicted=p_best[worst_idx], abs_error=err[worst_idx])
elif IS_ML:
    lab = to_label(p_best)
    nerr = (lab != EVAL_Y).sum(axis=1)
    by_len = pd.Series(nerr > 0).groupby(bucket.values, observed=True).mean().rename('row_error_rate')
    p, r, f, s = precision_recall_fscore_support(EVAL_Y, lab, zero_division=0)
    report = pd.DataFrame({'precision': p, 'recall': r, 'f1': f, 'support': s}, index=CLASS_NAMES).sort_values('f1')
    report.to_csv(CFG.FINAL_DIR / 'classification_report.csv')
    display(report.round(3))
    worst_idx = np.argsort(-nerr, kind='stable')[:15]
    worst = EVAL_FRAME.iloc[worst_idx][['text']].assign(actual=[ml_names(EVAL_Y[i]) for i in worst_idx],
                                                        predicted=[ml_names(lab[i]) for i in worst_idx], label_errors=nerr[worst_idx])
else:
    lab = to_label(p_best)
    by_len = pd.Series(lab != EVAL_Y).groupby(bucket.values, observed=True).mean().rename('error_rate')
    p, r, f, s = precision_recall_fscore_support(EVAL_Y, lab, labels=list(range(N_CLASSES)), zero_division=0)
    report = pd.DataFrame({'precision': p, 'recall': r, 'f1': f, 'support': s}, index=CLASS_NAMES).sort_values('f1')
    report.to_csv(CFG.FINAL_DIR / 'classification_report.csv')
    display(report.round(3))
    conf = p_best.max(axis=1)
    wrong = np.where(lab != EVAL_Y)[0]
    worst_idx = wrong[np.argsort(-conf[wrong])][:15]
    worst = EVAL_FRAME.iloc[worst_idx][['text']].assign(actual=[CLASS_NAMES[i] for i in EVAL_Y[worst_idx]],
                                                        predicted=[CLASS_NAMES[i] for i in lab[worst_idx]], confidence=conf[worst_idx])
worst['text'] = worst['text'].map(lambda t: short(t, 160))
worst.to_csv(CFG.FINAL_DIR / 'worst_predictions.csv', index=False)

fig, axes = plt.subplots(1, 2 if not IS_REG else 1, figsize=(16 if not IS_REG else 9, 4.8), squeeze=False)
labels_len = [f'{b}\n({int(edges.loc[b, "min"])}-{int(edges.loc[b, "max"])} w)' for b in by_len.index]
axes[0, 0].bar(labels_len, by_len.values, color=[PALETTE[i] for i in range(len(by_len))])
bar_labels(axes[0, 0], '{:.3f}')
axes[0, 0].set_title(f'{by_len.name} by document length', loc='left')
axes[0, 0].tick_params(axis='x', labelsize=8)
if IS_ML:
    axes[0, 1].barh([short(i, 18) for i in report.index], report['f1'], color=PRIMARY)
    axes[0, 1].set_title('Holdout F1 per label', loc='left')
elif not IS_REG:
    cm = pd.crosstab(pd.Series(EVAL_Y, name='a'), pd.Series(lab, name='p'))
    pairs = cm.stack().rename('n').reset_index()
    pairs = pairs[pairs['a'] != pairs['p']].nlargest(10, 'n')
    names = [f'{short(CLASS_NAMES[a], 14)} -> {short(CLASS_NAMES[b], 14)}' for a, b in zip(pairs['a'], pairs['p'])]
    axes[0, 1].barh(names[::-1], pairs['n'].values[::-1], color=ACCENT)
    axes[0, 1].set_title('Most frequent confusions (actual -> predicted)', loc='left')
suptitle(fig, 'Where does the final model fail?', f'{EVAL_NAME} set, {BEST_NAME}')
save_fig(fig, 'error_analysis', 'final')
display(worst.head(10))

#### Insights

> *Fill in after running.* Do long documents fail more often, which would argue for a larger `MAX_LENGTH`, a long-context model such as ModernBERT, or chunking? Which class pairs are confused, and are the most confident errors genuine mistakes or label noise?

---

# Predictions and Export <a name="13"></a>

---

The selected model's out-of-fold, holdout, and test predictions are written to `pipeline-output/final/`, with `submission.csv` when a test file is configured. The resolved configuration, the EDA decisions table, the leaderboard, and a markdown report complete the export. Fine-tuned weights are saved only when `SAVE_WEIGHTS = True`, because base-size checkpoints take about 0.5 to 1 GB each.

In [ ]:
export_result(BEST, CFG.FINAL_DIR)
settings = {k: v for k, v in vars(Settings).items() if not k.startswith('__') and not callable(v)}
resolved = {'task': TASK, 'metric': METRIC, 'language': LANGUAGE, 'max_length': MAX_LENGTH,
            'tfidf_max_features': TFIDF_MAX_FEATURES, 'class_weights': USE_CW, 'dedup': DEDUP,
            'group_identical': GROUP_IDENTICAL, 'models': {k: CHECKPOINT[k] for k in ACTIVE},
            'train_folds': TRAIN_FOLDS, 'best': BEST_NAME, 'weights': WEIGHTS, 'device': DEVICE, 'libraries': AVAILABLE,
            'decision_rule': DECISION_RULE, 'threshold': THRESHOLD[0], 'class_scale': CLASS_SCALE[0], 'cuts': CUTS[0],
            'label_thresholds': ML_THRESH[0], 'class_names': CLASS_NAMES}
to_json({'settings': settings, 'resolved': resolved, 'eda_decisions': EDA}, CFG.FINAL_DIR / 'config.json')

fmt = lambda v: f'{v:.4f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else '-'
cols = [f'cv_{METRIC}_shared', f'cv_{METRIC}_all', f'holdout_{METRIC}']
lines = [f'# NLP Modelling Report: {DATA_PATH.name}', '', '## Setup',
         f'- Task: **{TASK}** on `{TEXT_COL}`' + (f' + `{CFG.TEXT_PAIR_COL}`' if CFG.TEXT_PAIR_COL else '') + f', target `{LABEL_COL}`'
         + (f' ({N_CLASSES} classes, imbalance {IMBALANCE:.1f}x)' if (not IS_REG and IMBALANCE) else f' ({N_CLASSES} ordered classes)' if IS_ORD else '')
         + (f'; decision rule tuned on OOF: {DECISION_RULE}' if DECISION_RULE else ''),
         f'- Rows: {len(dev):,} dev, {len(hold):,} holdout' + (f', {len(test):,} test' if test is not None else ''),
         f'- Validation: {CFG.N_FOLDS}-fold' + (' grouped by identical text' if GROUP_IDENTICAL else '') + f'; transformers trained on folds {TRAIN_FOLDS}',
         f'- Metric: **{METRIC}**; max_length {MAX_LENGTH} ({CFG.TRUNCATION}); class weights {USE_CW}', '',
         '## Decisions', '', '| setting | value | source |', '|---|---|---|']
lines += [f'| {s} | `{r.value}` | {r.source} |' for s, r in decision_table.iterrows()]
lines += ['', '## Leaderboard', '', '| model | ' + ' | '.join(cols) + ' |', '|---' * (len(cols) + 1) + '|']
for m, r in LEADERBOARD.iterrows():
    lines.append(f'| {m}{" (selected)" if m == BEST_NAME else ""} | ' + ' | '.join(fmt(r[c]) for c in cols) + ' |')
lines += ['', f'## Final result ({EVAL_NAME})',
          f'- {BEST_NAME}: {METRIC} = {FINAL["scores"][METRIC]:.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f})',
          f'- TF-IDF baseline: {FINAL["baseline_scores"][METRIC]:.4f}' + (f'; gain CI {ci_gain[0]:+.4f} to {ci_gain[1]:+.4f}' if BEST_NAME != 'tfidf' else ''),
          '- All metrics: ' + ', '.join(f'{k} {v:.4f}' for k, v in FINAL['scores'].items())]
if WEIGHTS:
    lines.append('- Ensemble weights: ' + ', '.join(f'{k} {v:.2f}' for k, v in WEIGHTS.items()))
report_md = '\n'.join(lines)
(CFG.FINAL_DIR / 'report.md').write_text(report_md, encoding='utf-8')
display(Markdown(report_md))

The last cell prints the output tree for both stages.

In [ ]:
for stage in (CFG.BASELINE_DIR, CFG.FINAL_DIR):
    print(f'{stage}/')
    for p in sorted(stage.rglob('*')):
        if p.is_file():
            print(f'   {str(p.relative_to(stage)):<50} {p.stat().st_size / 1e3:>10,.1f} KB')

#### Insights

> *Fill in after running.* Summarise the chosen model and why (EDA evidence for the encoder and sequence length), its holdout score with confidence interval, the gain over TF-IDF, the words that drive predictions, and the main failure mode. These points form the conclusion.